# 02 · Forecasting Experiments
**Goal:** predict next weeks' demand and find out which simple model is best.

Golden rules we follow:
- **Time-based split** — train on the past, test on the newest weeks (never shuffle time data).
- Always beat a **baseline** (Seasonal Naive) or the fancy model isn't worth it.

In [ ]:
import sys
sys.path.append('..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.features import to_wide
from src.forecasting import MODELS, forecast
from src.evaluation import time_split, mae, rmse, wape

plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})
weekly = pd.read_csv(config.DATA_PROCESSED / 'weekly_demand.csv', parse_dates=['Week'])
wide = to_wide(weekly)
wide.shape

## Step 1 · Split the timeline

In [ ]:
train, test = time_split(wide, config.TEST_WEEKS)
print('Train:', train.index.min().date(), '->', train.index.max().date(), f'({len(train)} weeks)')
print('Test: ', test.index.min().date(), '->', test.index.max().date(), f'({len(test)} weeks)')

## Step 2 · Try all 5 models on ONE product

In [ ]:
code_ = wide.sum().idxmax()       # best-selling product
name = weekly.loc[weekly.StockCode == code_, 'Description'].iloc[0].title()
print(name)

fig, ax = plt.subplots()
ax.plot(wide.index, wide[code_], color='black', lw=2.5, label='Actual')
rows = []
for model_name in MODELS:
    preds = forecast(model_name, train[code_].values, len(test))
    ax.plot(test.index, preds, ls='--', label=model_name)
    rows.append([model_name, mae(test[code_], preds), rmse(test[code_], preds), wape(test[code_], preds)])
ax.axvspan(test.index.min(), test.index.max(), color='grey', alpha=.1)
ax.set_title(f'{name}: forecasts vs actual', loc='left', fontweight='bold')
ax.legend(frameon=False, ncol=3)
plt.show()

pd.DataFrame(rows, columns=['Model', 'MAE', 'RMSE', 'WAPE']).sort_values('WAPE').round(2)

## Step 3 · Score ALL products (this is what `run.py` saved)

In [ ]:
metrics = pd.read_csv(config.DATA_PROCESSED / 'model_metrics.csv')
metrics

In [ ]:
ax = metrics.plot(kind='bar', x='Model', y='WAPE', legend=False, color=['#10B981'] + ['#9CA3AF'] * (len(metrics) - 1))
ax.set_title('WAPE by model (lower is better)', loc='left', fontweight='bold'); ax.set_xlabel('')
plt.xticks(rotation=20)
plt.show()

## Step 4 · Mini-experiment: how long should a moving average be?
Try windows from 2 to 12 weeks and see which one wins.

In [ ]:
from src.forecasting import moving_average
results = {}
for window in range(2, 13):
    errors_a, errors_p = [], []
    for c in wide.columns:
        preds = np.repeat(np.mean(train[c].values[-window:]), len(test))
        errors_a += list(test[c]); errors_p += list(preds)
    results[window] = wape(errors_a, errors_p)
pd.Series(results, name='WAPE').plot(marker='o', color='#4F46E5')
plt.title('Moving-average window vs error', loc='left', fontweight='bold')
plt.xlabel('Window (weeks)'); plt.show()

## ✍️ Conclusions (write yours here!)
- Which model won, and by how much vs the baseline?
- Did the machine-learning model beat the simple ones? **Why / why not?** (Hint: we only have two years of data.)
- What would you try next? (more history, holiday flags, promotions, price)